# COCO2017学习笔记：先看模型怎么走，再看代码怎么写

**这组实验要回答：YOLO11s剪小以后，普通训练能恢复多少精度，加入教师又能额外恢复多少？**

先读下面的“主流程”，看清输入、输出和分支；读懂后再跳到对应代码。后半部分保留完整实验记录，供查原理、命令和来源。历史实验编号不是必须从头执行的操作步骤。

- [① 最终流程：从官方模型到结果](#flow-overview)
- [② 每步拿什么、做什么、交给谁](#flow-steps)
- [③ 跟着node15走一次](#flow-node15)
- [④ 怎样从同一source公平比较四种训练](#flow-training)
- [⑤ 数据和命令按什么顺序用](#flow-data-commands)
- [⑥ 历史尝试为什么出现](#flow-history)
- [原理、代码、命令详细索引](#detail-index)

本笔记整理已有实验，没有重新训练。代码单元保留源文件、函数、行号与哈希，用于阅读；启动实验应使用完整脚本的终端指令。


<a id="flow-overview"></a>

## ① 先看最终流程：两条线，一次比较

**第一条线负责造出不同大小的模型；第二条线负责比较怎样把它们训练好。**

~~~text
【造模型：raw剪枝主链】
官方YOLO11s
   │ 真实敏感度分析：找出哪些层更能承受删通道（实验16）
   ▼
按层级限额剪到5%（实验17）
   ▼
raw5 ──继续删通道──> raw10 ──继续删通道──> raw15 ──继续删通道──> raw20
 │                    │                    │                    │
 └─取副本做BN校准───────┴────────────────────┴────────────────────┘
                      ▼
            source5 / source10 / source15 / source20
                      │
【比较训练方法：每个方法重新从同一source开始】
                      ├─普通检测训练 control
                      └─检测训练 + YOLO11s教师 KD
                             │
                      每条分支各自选权重、测val（19B，共8任务）
                             ▼
                      据已有结果选出node15和node20
                             │
             回到原source15 / source20，再各自新增两条训练分支
                      ├─检测训练 + YOLO11m教师 KD
                      └─检测训练 + DINOv2教师 KD（19C，共4任务）
                             ▼
                 汇总12任务：比较精度、计算量、延迟与训练成本
~~~

读箭头时记住：

1. **只有raw继续剪。** BN版、control结果和KD结果都不送回raw主链。
2. **control与KD各自从同一source起跑。** 本实验没有先训练control、再拿control的best继续蒸馏。
3. **KD本身就是一次恢复训练。** 每批图像同时计算检测损失和蒸馏损失，不是先训练完检测再补一次KD。
4. **每到一个节点，就在副本上做BN和评价。** 图中合画BN支路是为了看清关系，不表示实际等所有节点剪完才校准。

上图表示模型依赖关系，不规定GPU上的并发时间。19C是看过19B的val后才选择两个节点，因此后面的val排名带有事后选型影响。

依据：[最终核验报告 §2、6、7](../reports/experiment20_COCO2017final_comparison.md)与[阶梯脚本](../scripts/prune_taylor_ladder_coco2017.py)。


<a id="flow-steps"></a>

## ② 每一步的输入 → 操作 → 输出

| 步骤 | 拿什么进来 | 具体做什么 | 得到什么，下一步用在哪里 |
|---|---|---|---|
| A 准备 | 官方YOLO11s + train2017/val2017 | 核对权重哈希；划分calibration、tune、recovery | 固定起点与数据清单，供所有后续阶段共用 |
| B 测层敏感度（16） | 官方模型 + calibration/tune | calibration算Taylor；每层独立试删8通道，在tune测掉点/省计算 | 层敏感度排序，供C设定low/medium/high限额 |
| C 建raw阶梯（17、19A） | 官方模型 + 层排序 | 先剪到5%；再沿raw链累计剪到10%、15%、20% | 四个不同大小的raw模型，分别交给D |
| D 选训练起点（BN） | 每个raw的副本 + recovery图像 | 只更新BN统计；在tune比较raw和BN版 | 本次四节点均选择BN版，命名为source |
| E 做训练对照（19B） | 四个source，各复制两份 | 一份control、一份YOLO11s KD，各20轮 | 8次训练，各自产生best/last并按下述规则选模型 |
| F 增加教师比较（19C） | 原source15、原source20 | 每个再做YOLO11m KD、DINOv2 KD，各20轮 | 4次新增训练；不是从E训练后的权重接着训 |
| G 汇总评价 | 各任务选中的模型 | 同口径看val、GMAC、参数、延迟、训练成本 | 判断本次压缩/恢复折中，而非只挑一个最大mAP |

**E和F中的每个任务都走相同的内部步骤：**

~~~text
source ──训练20轮──> best / last
   │                    │
   │            KD分支先清理教师/投影，得到best_clean / last_clean
   └────────────────────┤
                        ▼
            source、best、last都在同一个tune上评价
                        ▼
                 选tune mAP最高的一个
                        ▼
                 只对选中模型测完整val2017
~~~

因此“完成20轮”和“最终采用训练后模型”不等价。5%/10%确实训练了，最终选回source；15%/20%选中训练best。

细节跳转：[数据划分](#chapter-1) · [敏感度与Taylor](#chapter-3) · [BN/raw阶梯](#chapter-4) · [训练实现](#chapter-5) · [模型选择代码](#chapter-7)。


<a id="flow-node15"></a>

## ③ 只跟着node15走一次：模型到底变了几次

不要先记所有实验编号。先把下面五种模型状态区分开：

| 模型状态 | 从哪里来 | 这次改变的内容 |
|---|---|---|
| 官方YOLO11s | 官方预训练权重 | 起点；参数9,458,752 |
| node15 raw | 官方→raw5→raw10→raw15 | 通道真的被删；结构变小，参数8,063,896 |
| node15 source | node15 raw的BN校准副本 | 仅BN统计改变；参数个数、通道宽度不变 |
| node15 control best | source独立普通训练20轮 | 更新学生参数，用检测标签恢复 |
| node15 YOLO11s KD best_clean | 同一source独立KD训练20轮后清理 | 更新学生与训练期投影；清理后只保留学生 |

source是“开始恢复训练的模型”的角色名，raw是“刚剪完未恢复”的状态名；本次source恰好是BN校准版。

**先单独看BN这一步（tune成绩）：**

~~~text
node15 raw 0.195039 ──BN统计校准──> node15 source 0.311249
~~~

**再看从source出发的训练比较（下面全部是val成绩）：**

~~~text
                           ┌─ control：检测损失 ─────────> 0.423467
node15 source 0.258892 ─────┤
                           └─ YOLO11s KD：检测损失+KD ──> 0.427374
~~~

两条支路输入是同一份source：

- 普通恢复收益：(0.423467−0.258892)×100 = **16.4575 AP点**。
- KD相对control的额外收益：(0.427374−0.423467)×100 = **0.3907 AP点**。

这解释了“恢复主要来自普通训练，KD额外提升较小”。两条结果的差值用于对照，**不代表0.423467的control模型被继续训练成0.427374**。

source在tune为0.311249、在val为0.258892，是同一模型在两个集合的成绩，不能把两者相减当退化。node15 raw没有完整val记录，不能补画raw→source的val收益。

历史文件对应：

~~~text
raw:
  runs/prune/experiment20_taylor_ladder_coco2017/20260917_023832/stage_15/raw.pt
source:
  runs/prune/experiment20_taylor_ladder_coco2017/20260917_023832/stage_15/bn_m0005_8192.pt
YOLO11s KD选中学生:
  runs/recovery/experiment19b_coco2017/node15_yolo11s_kd/20260917_101816/training/weights/best_clean.pt
~~~

路径以历史服务器/root/YOLO为根；本地不一定含这些权重。数值来自[最终核验报告 §4–6](../reports/experiment20_COCO2017final_comparison.md)。


<a id="flow-training"></a>

## ④ 一次训练内部做什么：control与KD的差异

| 对照项 | control | YOLO11s / YOLO11m / DINOv2 KD |
|---|---|---|
| 初始学生 | 对应node的同一source | 对应node的同一source |
| 输入图像与标签 | recovery | recovery |
| 学生计算 | 图像→预测→检测损失 | 同左，再取P3/P4/P5特征 |
| 教师计算 | 无 | 冻结教师看同一批图像，提供特征目标 |
| 总损失 | 检测损失 | 检测损失 + λ×特征差异 |
| 梯度更新 | 学生 | 学生 + 投影层；教师不更新 |
| 训练结束 | best/last | wrapper先移除教师和投影，得到clean学生 |
| 选择标准 | source/best/last在tune上比较 | 相同，使用清理后的学生比较 |

**理解一批图像的先后顺序：**

~~~text
图像 + 真实框/类别 ──> 学生预测 ──> 检测损失 ────────┐
                    └─> 学生特征 ──> 投影 ──┐       │
同一图像 ──> 冻结教师 ──> 教师特征 ────────> KD损失 ─┤
                                                   ▼
                                           合并损失、反向传播
                                                   ▼
                                          更新学生和投影层参数
~~~

control就是保留检测损失这条路。KD增加教师监督，学生仍然学习真实标签。

接下来看代码只需抓住四个入口：

1. [train_control](../scripts/recover_ladder_control_coco2017.py)：把已剪学生放入训练器，保留结构。
2. [YoloFeatureDistiller.loss](../scripts/recover_ladder_yolo_kd_coco2017.py)：比较学生与教师的特征。
3. [YoloKDStudentModel.loss](../scripts/recover_ladder_yolo_kd_coco2017.py)：把检测损失与KD项合起来。
4. [detach_for_inference](../scripts/recover_ladder_yolo_kd_coco2017.py)：训练后去掉教师、投影和hook。

源代码逐段摘录见[第五部分](#chapter-5)和[第六部分](#chapter-6)。


<a id="flow-data-commands"></a>

## ⑤ 数据和脚本怎么接：照输出找下一步输入

| 当前阶段 | 用哪份数据 | 有没有更新学生权重 |
|---|---|---|
| Taylor通道评分 | calibration图像与标签 | 只算梯度，不做优化器更新 |
| 层敏感度评价 | tune | 只评估试剪副本 |
| raw结构剪枝 | calibration提供评分，副本统计GMAC | 删除通道；不做恢复训练 |
| BN校准 | recovery中8192张图像 | 仅更新BN统计，不更新可训练参数 |
| control/KD训练 | recovery图像与标签 | 更新学生；KD还更新投影层 |
| 选source/best/last | tune | 不更新 |
| 选中模型完整评价 | val2017 | 不更新，但本次还用其结果作节点/教师事后比较 |

**这是依赖顺序，不是让你把笔记所有命令从上到下一次执行。**

| 顺序 | 脚本 | 关键输出 → 下一步输入 |
|---|---|---|
| 1 | measure_taylor_sensitivity_coco2017.py | 完成的敏感度目录 → SENS；内部含排序、清单和recovery.yaml |
| 2 | prune_taylor_tiered_coco2017.py | 读取--sensitivity-run SENS → 5% raw运行目录STAGE5 |
| 3a | recalibrate_bn_coco2017.py | 读取STAGE5与SENS → 5% BN方案比较、source5 |
| 3b | prune_taylor_ladder_coco2017.py | 读取原STAGE5 raw与SENS → stage_10/15/20，每个含raw和BN副本 |
| 4 | recover_ladder_control_coco2017.py | --source选某node的BN版 → control训练结果 |
| 5 | recover_ladder_yolo_kd_coco2017.py | --source仍选同一BN版，额外指定--teacher → YOLO KD结果 |
| 6 | recover_ladder_dinov2_kd_coco2017.py | --source仍选同一BN版 → DINO KD结果 |

3a不把BN版传给3b；4不把control结果传给5或6。

Bash变量和“用哪个文件”的对应关系：

~~~bash
# 来源：scripts/run_experiment19b.sh:13–35、恢复脚本parse_args
# 假设已在/root/YOLO，并按环境章节设置PY和DATA
SENS=runs/analysis/experiment16_taylor_sensitivity_coco2017/20260915_122417
STAGE5=runs/prune/experiment17_tiered_taylor_coco2017/20260915_123525
LADDER=runs/prune/experiment20_taylor_ladder_coco2017/20260917_023832
SRC15="$LADDER/stage_15/bn_m0005_8192.pt"
RECOVERY="$SENS/recovery.yaml"

# 相同SRC15，两条独立分支；以下只做预检
"$PY" scripts/recover_ladder_control_coco2017.py --node 15 --source "$SRC15" --data "$RECOVERY" --dataset-root "$DATA"
"$PY" scripts/recover_ladder_yolo_kd_coco2017.py --node 15 --source "$SRC15" --teacher weights/yolo11s.pt --teacher-name yolo11s --data "$RECOVERY" --dataset-root "$DATA"
~~~

完整20轮命令见[第五部分第18节](#chapter-5)。这些变量指向已有历史文件；新跑得到新时间戳时需更新依赖路径，且恢复脚本有source哈希约束。原脚本中“实验20”文件夹实际上服务19A，不要另找一个“20轮剪枝步骤”。


<a id="flow-history"></a>

## ⑥ 历史尝试放在哪里：解释为什么这样设计

**最终流程是上面的A→G。下面是设计的来历，不是必须先跑完的前置流水线。**

| 历史尝试 | 发现的问题/作用 | 对最终流程的影响 |
|---|---|---|
| 07 官方验证 | 知道官方模型的精度与计算量起点 | 先核对基线 |
| 08 继续训练尝试 | 当前材料缺完整完成记录 | 不编造成绩，也不将其自动接成09父权重 |
| 09/10 L1探测与贪心剪枝 | 20步仅省1.19% GMAC，微调未改善 | 更重视实际GMAC收益与恢复对照 |
| 14 模型谱系/训练诊断 | 父权重不一致，同配方也会伤害未剪父模型 | 统一官方起点、用source保底 |
| 15 独立10% Taylor路径 | raw掉点大，恢复后仍低于官方 | 采用16的层敏感度和17的温和起点继续探索 |
| 18 5%保守恢复 | 低学习率训练也没超过raw | 保留负结果，进一步检查BN适配 |
| BN扫描 | 校准可改善训练前状态 | 四节点统一BN配方作为source |

**实验15的10%模型不传给19A的node10；实验18训练后的best不传给19A的raw5。** 最终raw链从实验17的原始5%剪枝模型开始。

只想学最终流程：读①–⑤，再查[BN/阶梯](#chapter-4)、[普通恢复/YOLO KD](#chapter-5)、[DINO](#chapter-6)、[选择评价](#chapter-7)。
想复盘“为什么改方法”：再读[07–10历史尝试](#chapter-2)和[14–18诊断过程](#chapter-3)。

来源：[实验14–15报告](../reports/experiment14_coco2017_recovery_plan.md)、[实验18报告](../reports/experiment18_tiered_taylor_recovery.md)、[最终核验报告](../reports/experiment20_COCO2017final_comparison.md)。


<a id="detail-index"></a>

## 详细资料索引（需要时再查）

**章节导航**

- [一、通用准备（先建立完整图景）](#chapter-1)
- [二、早期尝试：基线、敏感度与贪心剪枝（07–10）](#chapter-2)
- [三、重新建立正式主线（14–18）](#chapter-3)
- [四、BN校准与嵌套剪枝阶梯（19A）](#chapter-4)
- [五、普通恢复与YOLO教师蒸馏（19B/19C）](#chapter-5)
- [六、DINOv2跨架构教师与部署权重](#chapter-6)
- [七、结果怎样读：选择、增益归因与速度](#chapter-7)
- [八、终端复现、文件结构与排错](#chapter-8)
- [九、快速问答（自己复述原理）](#chapter-9)
- [十、结论与下一步学习](#chapter-10)
- [十一、报告与代码索引](#chapter-11)


<a id="chapter-1"></a>

## 一、通用准备（先建立完整图景）

### 1. 本部分作为详细资料查询

先读前面的[主流程](#flow-overview)理解模型依赖，再查本部分的原理和代码。实验编号按历史出现次序排列，不能据此把所有产物串行传给下一次训练。

### 2. 数据怎么分，分别用于什么

| 集合 | 图像数 | 来源 | 本项目用途 |
|---|---:|---|---|
| calibration | 2,048 | train2017 | 带检测标签计算Taylor梯度分数 |
| tune | 2,048 | train2017 | 敏感度排序、BN方案、source/best/last选择 |
| recovery | 114,191 | train2017剩余图像 | 普通恢复与KD训练；BN扫描也从这里取图 |
| val2017 | 5,000 | 官方验证集 | 已选权重完整检测评价 |

三份train子集互不重叠，总计118,287张。正式BN配方取recovery中8192张，它与Taylor的2048张校准集不同。

recovery.yaml的train指recovery，val指tune；完整configs/coco2017.yaml的val才指val2017。要读字段，不能只看文件名。不同集合的mAP不能直接相减。

来源：[实验16报告](../reports/experiment16_taylor_sensitivity_coco2017.md)、[最终核验报告 §2–3](../reports/experiment20_COCO2017final_comparison.md)。


**关键代码：固定种子后，一次排列切成三个互斥集合**

来源：[scripts/measure_taylor_sensitivity_coco2017.py](../scripts/measure_taylor_sensitivity_coco2017.py)，局部代码段，第 95–105 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/measure_taylor_sensitivity_coco2017.py:95–105
# 对应：局部代码段
# 文件 SHA256：e323fcd14ecc9e7551007a1195603ff3c4823dce5620ad8e818f993a807bbad5
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

order = torch.randperm(
    len(lines), generator=torch.Generator().manual_seed(args.seed)
).tolist()
calibration = [lines[i] for i in order[: args.calibration_images]]
tune = [
    lines[i]
    for i in order[
        args.calibration_images : args.calibration_images + args.tune_images
    ]
]
recovery = [lines[i] for i in order[args.calibration_images + args.tune_images :]]


读法：同一order连续切片，图像不会落入两个集合；固定种子还要求输入清单及顺序一致。原脚本保存清单与SHA256，迁移后要核对。

### 3. 环境与路径：本地阅读、服务器运行

当前Windows仓库为 C:\Users\22565\Desktop\YOLO_Primary，实际目录名是YOLO_Primary。本地数据配置指向 C:/Users/22565/datasets/coco。历史服务器仓库为 /root/YOLO，数据为 /root/datasets/coco。

最终报告记录A800-SXM4-80GB、Python 3.12.11、PyTorch 2.13.0+cu132、Ultralytics 8.4.144；复核计数使用Torch-Pruning 1.6.1、ultralytics-thop 2.1.6。实验15曾记录Torch-Pruning 1.6.0。不同阶段记录不能拼成完整历史版本锁定文件。

**PowerShell：本地只读检查。** .venv不存在时换为实际安装依赖的Python路径。

~~~powershell
Set-Location -LiteralPath 'C:\Users\22565\Desktop\YOLO_Primary'
Test-Path -LiteralPath '.\.venv\Scripts\python.exe'
Get-Content -LiteralPath '.\requirements-coco2017.txt'
Get-Content -LiteralPath '.\configs\coco2017.yaml' -TotalCount 8
Get-FileHash -LiteralPath '.\weights\yolo11s.pt' -Algorithm SHA256
# 仅在上面的Python存在时运行
.\.venv\Scripts\python.exe -m pip show torch ultralytics torch-pruning ultralytics-thop
.\.venv\Scripts\python.exe scripts\prune_taylor_coco2017.py --help
~~~

**Bash：历史服务器环境入口。** 后续正式命令均从这里开始；新服务器须改为实际环境与数据路径。

~~~bash
cd /root/YOLO
PY=/usr/local/miniconda3/envs/py312/bin/python
DATA=/root/datasets/coco
"$PY" -m pip show torch ultralytics torch-pruning ultralytics-thop
nvidia-smi
~~~

环境检查命令是整理新增；路径来源为scripts/run_experiment19b.sh:8–10和configs/coco2017.yaml:2–5。batch=128是A800正式配方，换硬件/配方后的结果应单独记录。

### 4. 模型谱系与指标词典

| 名称 | 实验中的含义 |
|---|---|
| official / parent | 官方未剪权重 / 剪枝模型的实际父权重，需用哈希确认 |
| raw | 结构化剪枝后、尚未BN校准或恢复 |
| BN / source | 恢复训练的起点；实验19四节点均选BN版 |
| control | 只用检测标签恢复训练，用于对照KD |
| best / last | 训练中最佳 / 最后一轮权重；best可能比source差 |
| clean checkpoint | 已移除教师、投影和特征hook的学生权重 |
| node15 | 累计计算量削减约15%的模型节点，不是第15层 |

Precision关心预测框中正确的比例，Recall关心真实目标被找回的比例；AP综合精确率-召回率曲线。mAP50以IoU=0.50匹配，mAP50-95平均多个IoU阈值，更严格。这里mAP用0–1，差值乘100才叫“AP点”。

GMAC是十亿次乘加，参数量是参数元素个数。计算量减少率=(基线GMAC−学生GMAC)/基线GMAC。有的FLOP约定把一次乘加算两次运算，必须注明口径。实际延迟还取决于硬件、精度、batch和实现。

谱系来源：[实验14–15报告](../reports/experiment14_coco2017_recovery_plan.md)。官方SHA256为85a76fe8…d502d5，实验09/10父权重为336469c5…5b1；两者不能混作同源对照。


<a id="chapter-2"></a>

## 二、早期尝试：基线、敏感度与贪心剪枝（07–10）

### 5. 实验07：先知道官方模型的起点

**原理**：在val2017直接验证官方COCO预训练YOLO11s。复制得到的baseline.pt是原权重快照，不是重新训练的模型。

**命令（本地旧入口，立即验证，可能下载缺失数据/权重）**：

~~~powershell
# 来源：scripts/run_yolo11s_coco2017_baseline.py -> main / run
.\.venv\Scripts\python.exe scripts\run_yolo11s_coco2017_baseline.py --device 0 --batch 32
~~~

内部检查本地数据清单路径，不能不加修改就作为服务器通用入口；会生成运行目录并重写实验07摘要报告。


**关键代码：YOLO.val的数据、分辨率、batch与输出目录**

来源：[scripts/run_yolo11s_coco2017_baseline.py](../scripts/run_yolo11s_coco2017_baseline.py)，局部代码段，第 120–126 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/run_yolo11s_coco2017_baseline.py:120–126
# 对应：局部代码段
# 文件 SHA256：9260ddb08e91b766a4994d2a449f5b2bccd3d6a5a1e3d6641ab46d3a13686788
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

model = YOLO(str(downloaded))
metrics = model.val(
    data=str(config), split="val", imgsz=640, batch=args.batch,
    device=args.device, workers=0, plots=True, verbose=True,
    seed=42, deterministic=True, project=str(run_dir), name="validation",
    exist_ok=False,
)


**结果与理解**：mAP50=0.6319，mAP50-95=0.4635，参数9,458,752，GMAC=10.7991。后续服务器官方记录约0.4633，应分别注明出处。验证器2.938 ms/图与batch=1前向中位数11.458 ms来自不同协议。

来源：[实验07报告](../reports/experiment07_yolo11s_coco2017_baseline.md)。

### 6. 实验08：继续训练成熟的预训练模型

**原理**：从官方权重继续更新参数，保存best/last。预算是配置，是否完成及精度怎样必须看结果文件。继续训练不保证提高成熟预训练模型的精度。

**命令（立即训练，无--execute开关）**：

~~~powershell
# 来源：scripts/train_yolo11s_coco2017.py -> main
.\.venv\Scripts\python.exe scripts\train_yolo11s_coco2017.py
~~~

超参数写在main中，不支持任意添加--epochs等CLI参数。当前配置30轮、imgsz=512、batch=64、patience=10，可能提前停止；脚本会写实验08报告。


**关键代码：实验08的固定训练配置**

来源：[scripts/train_yolo11s_coco2017.py](../scripts/train_yolo11s_coco2017.py)，局部代码段，第 45–49 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/train_yolo11s_coco2017.py:45–49
# 对应：局部代码段
# 文件 SHA256：0d0ea952d53c2b41cdc0a191cfedb9d01c5dadf58fcb80072673193c16a0f8b2
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

settings = {
    "model": str(weights), "data": str(config), "epochs": 30,
    "imgsz": 512, "batch": 64, "device": 0, "workers": 2,
    "seed": 42, "deterministic": False, "pretrained": True, "patience": 10,
}


**记录边界**：本次检查到多个实验08目录和train_config.json，但未找到这些目录中的results.csv或train_info.json，也没有顶层实验08完成报告。这里只总结脚本意图，不编造最终成绩；不能只凭名称认定实验09/10父权重来自某一次实验08。

### 7. 实验09：逐层L1 masking敏感度

**原理**：每次重载相同父模型，只将一个卷积层中L1幅值最低的一部分滤波器置零，测mAP下降。flatten(1)展平每个输出滤波器，mean(1)得到每通道分数，升序选择待屏蔽通道。

张量大小没有改变，所以不减少GMAC；后接BN时，零卷积输出也不保证最后特征为零。

**当前本地实现的机制学习命令**（立即评估，须先补齐同一父权重）：

~~~powershell
# 来源：scripts/prune_sensitivity.py -> main
# 使用新的CSV名保留历史结果
.\.venv\Scripts\python.exe scripts\prune_sensitivity.py --weights weights/baseline/yolo11s_coco2017_best.pt --data configs/coco2017.yaml --ratio 0.10 --device 0 --output reports/experiment09_learning_recheck.csv
~~~

历史服务器报告用batch=128、workers=8；当前validate()写死batch=8、workers=0，CLI没有这些参数。因此这是当前实现重跑命令，不是完整历史复现。


**关键代码：按L1排序后屏蔽滤波器**

来源：[scripts/prune_sensitivity.py](../scripts/prune_sensitivity.py)，mask_low_l1_filters，第 45–55 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/prune_sensitivity.py:45–55
# 对应：mask_low_l1_filters
# 文件 SHA256：fff10fd7d682d94c8e499d6ef76fd1ab114724149a189cbf812440a4454d14ee
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

def mask_low_l1_filters(conv: torch.nn.Conv2d, ratio: float) -> int:
    """将一个卷积层中 L1 重要性最低的部分输出通道置零。"""
    count = max(1, round(conv.out_channels * ratio))
    count = min(count, conv.out_channels - 1)
    scores = conv.weight.detach().abs().flatten(1).mean(1)
    indices = torch.argsort(scores)[:count]
    with torch.no_grad():
        conv.weight[indices] = 0
        if conv.bias is not None:
            conv.bias[indices] = 0
    return count


**结果**：87层，屏蔽比例10%，父模型val mAP50-95=0.4450059486。drop大表示本测试下较敏感；微小负drop不证明稳定改善。检测头输出有类别/回归语义，需要额外结构保护。

来源：[实验09报告](../reports/experiment09_yolo11s_coco2017_sensitivity.md)。

### 8. 实验10：贪心真实删通道，为什么没达到目标

**原理**：在当前模型副本上试剪，依赖图同步删除相关张量，再评估mAP。按“本步精度损失/本步计算量节省”选动作。代码中的节省量是相对原始基线的比例；同一基线下与绝对GMAC节省归一化给出的排序等价。

**命令（无预检开关，依赖实验09原父权重和敏感度CSV）**：

~~~bash
# 来源：scripts/prune_greedy_coco2017.py -> main
"$PY" scripts/prune_greedy_coco2017.py --weights weights/baseline/yolo11s_coco2017_best.pt --data configs/coco2017.yaml --sensitivity reports/experiment09_yolo11s_coco2017_sensitivity.csv --target-reduction 0.20 --max-steps 20 --epochs 10 --device 0 --run-label experiment10_learning_recheck
~~~


**关键代码：评价试剪候选，选择较小的精度/计算量代价**

来源：[scripts/prune_greedy_coco2017.py](../scripts/prune_greedy_coco2017.py)，局部代码段，第 246–262 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/prune_greedy_coco2017.py:246–262
# 对应：局部代码段
# 文件 SHA256：f271b956d3932569700e86e8766052583fa39f62fda5dada655c49740789ab09
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

gain = (current_gmacs - gmacs) / baseline_gmacs
if gain <= 1e-6:
    raise ValueError('no_compute_reduction')
save_model(trial, run_dir / '_candidate.pt', base)
measured = evaluate(run_dir / '_candidate.pt', data, run_dir / 'validation/candidate', args.device)
if not all(math.isfinite(float(x)) for x in measured.values()):
    raise ValueError('nonfinite_metric')
if original_eval['map50_95'] - measured['map50_95'] > args.max_map_drop:
    raise ValueError('cumulative_accuracy_drop_limit')
drop = current_map - measured['map50_95']
row.update(change)
row.update({'parameters': params, 'gmacs': gmacs, **measured, 'step_map_drop': drop,
    'step_compute_saved': gain, 'total_compute_saved': 1 - gmacs / baseline_gmacs,
    'score': drop / gain})
if best_row is None or (row['score'], -gain, name) < (best_row['score'], -best_row['step_compute_saved'], best_row['layer']):
    backward_check(trial, device)
    best_model, best_row = copy.deepcopy(trial), dict(row)


| 阶段 | val mAP50-95 | GMAC |
|---|---:|---:|
| 实际父模型 | 0.4450 | 10.7991 |
| 20步剪枝raw | 0.4374 | 10.6709 |
| 10轮微调best | 0.4288 | 10.6709 |

目标20%计算量削减，实际1.19%；参数减少1.95%。搜索多次选中低计算量贡献层，达到20步上限仍不够。微调继续下降，前向中位耗时5.421→5.441 ms也未改善。

**学习要点**：结构可用、训练完成和效果成功是三件事。后续需要改进计算量收益的考虑，并诊断恢复配方。

来源：[实验10报告](../reports/experiment10_coco2017_greedy.md)。


<a id="chapter-3"></a>

## 三、重新建立正式主线（14–18）

### 9. 实验14：核对父权重，诊断恢复训练

**原理**：对照实验应只改变研究因素。实验10来自0.4450父模型，拿0.4633官方模型直接当“剪前”会混入父权重差异。先用SHA256核对谱系，再让父模型/剪枝模型接受同一恢复配方。

| 预设 | 改什么 | 想回答什么 |
|---|---|---|
| legacy_frozen | 复现旧冻结BN设置 | 历史行为能否重现 |
| bn_update | 相对旧方案改为BN更新 | BN策略是否造成问题 |
| standard_recovery | BN更新、较大学习率、适度增强 | 常规恢复训练是否有效 |

**实验15产物的诊断命令模板**：目录需真实存在；移除--execute做预检。输入run_info的父哈希必须对应官方权重。

~~~bash
# 来源：scripts/diagnose_coco2017_recovery.py -> parse_args / verify_lineage
TAYLOR_RUN=runs/prune/experiment15_taylor_coco2017/20260915_020645
"$PY" scripts/diagnose_coco2017_recovery.py --parent weights/yolo11s.pt --pruned "$TAYLOR_RUN/pruned_raw.pt" --provenance "$TAYLOR_RUN/run_info.json" --dataset-root "$DATA" --preset standard_recovery --epochs 10 --batch 128 --val-batch 128 --device 0 --execute
~~~

这是按当前CLI整理的重跑模板，不声称是完整历史shell原文。

来源：[实验14–15报告](../reports/experiment14_coco2017_recovery_plan.md)。

### 10. 实验15：Taylor分数怎样衡量“剪谁”

**原理**：把权重删成0，相当于扰动Δw=−w。损失一阶近似与w×∂L/∂w有关。本项目对每个输出通道内的|w×grad|求和，再按图像数聚合为剪枝代价。这是启发式分数，不是剪后真实mAP下降的精确预测，也没有完整累计依赖组所有参数的代价。

收集分数需要标签和反向传播，但不调用optimizer.step；模型走训练损失分支，同时BN保持eval。计算梯度和更新权重可以分开。

~~~bash
# 来源：scripts/prune_taylor_coco2017.py -> parse_args / main
# 先预检
"$PY" scripts/prune_taylor_coco2017.py --dataset-root "$DATA"
# 正式运行：从官方权重独立开始，计算量目标减少10%
"$PY" scripts/prune_taylor_coco2017.py --dataset-root "$DATA" --target-reduction 0.10 --channel-step 8 --calibration-images 2048 --calibration-batch 32 --val-batch 128 --device 0 --execute
~~~


**关键代码：BN统计不变，同时允许计算梯度**

来源：[scripts/prune_taylor_coco2017.py](../scripts/prune_taylor_coco2017.py)，局部代码段，第 162–171 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/prune_taylor_coco2017.py:162–171
# 对应：局部代码段
# 文件 SHA256：430e0d307f448920497ac7310044f9944db1eb19f2c4c331d7faf19cc3272c8e
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

model = YOLO(str(base)).model.float().to(device)
before = fingerprint(model)
model.args = get_cfg(overrides=dict(model.args))
model.criterion = None
model.train()
for parameter in model.parameters():
    parameter.requires_grad_(True)
for module in model.modules():
    if isinstance(module, nn.BatchNorm2d):
        module.eval()


**关键代码：反向传播后按输出通道聚合Taylor分数**

来源：[scripts/prune_taylor_coco2017.py](../scripts/prune_taylor_coco2017.py)，局部代码段，第 184–208 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/prune_taylor_coco2017.py:184–208
# 对应：局部代码段
# 文件 SHA256：430e0d307f448920497ac7310044f9944db1eb19f2c4c331d7faf19cc3272c8e
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

for batch_index, batch in enumerate(loader, start=1):
    batch = {
        key: value.to(device, non_blocking=True)
        if isinstance(value, torch.Tensor)
        else value
        for key, value in batch.items()
    }
    batch["img"] = batch["img"].float() / 255.0
    size = int(batch["img"].shape[0])
    model.zero_grad(set_to_none=True)
    loss, _ = model(batch)
    loss = loss.sum() / size
    if not torch.isfinite(loss).item():
        raise RuntimeError("Taylor calibration produced non-finite loss")
    loss.backward()

    for name, total in scores.items():
        weight = modules[name].weight
        if weight.grad is None or not torch.isfinite(weight.grad).all().item():
            raise RuntimeError(f"Missing or non-finite gradient for {name}")
        # Sum over each filter is the first-order Taylor channel cost.
        value = (weight.detach() * weight.grad.detach()).abs().flatten(1).sum(1)
        total.add_(value.double().cpu(), alpha=size)

    image_count += size


**为什么还要除以GMAC收益**：代价小的动作可能省不了多少计算。每个副本真实剪完后统计GMAC，用Taylor代价/实际节省GMAC排序，小者优先。

实验15的search()使用开始时收集的分数，通过原始通道ID跟踪后续通道；实验17及19A改成每次接受动作后，在当前结构上重新收集分数。不能把“每步重算”概括到所有Taylor尝试。

**结果**：43步达到10.03% GMAC减少；官方0.4633→raw 0.1723→10轮恢复0.4144。未剪父模型同配方训练best仅0.4206，选择器保留官方source。这说明配方也会扰动成熟模型，不能把全部退化归因于剪枝。实验15的10%模型与后续node10结构不同。

来源：[实验14–15报告正式运行结果](../reports/experiment14_coco2017_recovery_plan.md)。

### 11. 依赖图：为什么不能只删卷积的一行

前一层输出通道被删，后一层对应输入通道、BN参数等也必须同步改变。残差相加、CSP分块、注意力和检测头还有额外约束。Torch-Pruning追踪依赖，项目仍需保护特定模块并检查输出形状。


**关键代码：建立依赖图并取得待剪组**

来源：[scripts/prune_taylor_coco2017.py](../scripts/prune_taylor_coco2017.py)，局部代码段，第 272–283 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/prune_taylor_coco2017.py:272–283
# 对应：局部代码段
# 文件 SHA256：430e0d307f448920497ac7310044f9944db1eb19f2c4c331d7faf19cc3272c8e
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

split_outputs = {
    module.cv1.conv
    for module in model.modules()
    if isinstance(module, (C2f, C2PSA))
}
before = shape_signature(model)
graph = tp.DependencyGraph().build_dependency(model, example_inputs=example)
group = graph.get_pruning_group(
    root, tp.prune_conv_out_channels, idxs=indices
)
if not graph.check_pruning_group(group):
    raise ValueError("dependency_group_rejected")


**关键代码：更新通道身份映射并执行整组剪枝**

来源：[scripts/prune_taylor_coco2017.py](../scripts/prune_taylor_coco2017.py)，局部代码段，第 325–335 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/prune_taylor_coco2017.py:325–335
# 对应：局部代码段
# 文件 SHA256：430e0d307f448920497ac7310044f9944db1eb19f2c4c331d7faf19cc3272c8e
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

updated = copy.deepcopy(identities)
for module_name, removed in output_deletions.items():
    updated[module_name] = [
        original_id
        for current_id, original_id in enumerate(identities[module_name])
        if current_id not in removed
    ]

before_channels = root.out_channels
removed_original_ids = [identities[name][index] for index in indices]
group.prune()


两段之间原脚本还有保护检查：C2f/C2PSA分块输出、stem、attention、DFL与检测头最终输出，以及最小剩余宽度。**不能将以上摘录拼成省略保护检查的生产剪枝器**；完整实现见prune_action()。

剪后检查320/640输入前向、有限值、反向传播、保存重载结构。前向通过不等于训练一定通过，结构有效也不等于精度达标。

### 12. 实验16：从masking升级为真实结构敏感度

**原理**：每一层都从同一官方权重开始，试删8个低Taylor通道，在tune上测mAP下降/实际GMAC节省。与实验09置零不同，也与后续累计剪枝不同。

~~~bash
# 来源：scripts/measure_taylor_sensitivity_coco2017.py -> parse_args
"$PY" scripts/measure_taylor_sensitivity_coco2017.py --dataset-root "$DATA" --calibration-images 2048 --tune-images 2048 --channel-step 8 --calibration-batch 32 --val-batch 128 --device 0 --execute
~~~

**结果**：42个候选，35个可行、7个被依赖保护拒绝。可行只说明这次试删满足条件，不代表可以大量剪。脚本生成后续共用清单与三个YAML。

来源：[实验16报告](../reports/experiment16_taylor_sensitivity_coco2017.md)。

### 13. 实验17：分档限额，先取得温和的5%节点

**原理**：按敏感度排序分low/medium/high。low上限12.5%，medium上限6.25%，high锁定；相对原通道数计算上限，再向下对齐8通道粒度。试剪后核查依赖联动是否让其他层超额。

~~~bash
# 来源：scripts/prune_taylor_tiered_coco2017.py -> parse_args
# 历史目录；重做实验16应改用新输出目录
SENS=runs/analysis/experiment16_taylor_sensitivity_coco2017/20260915_122417
"$PY" scripts/prune_taylor_tiered_coco2017.py --dataset-root "$DATA" --sensitivity-run "$SENS" --target-reduction 0.05 --low-cap 0.125 --medium-cap 0.0625 --channel-step 8 --device 0 --execute
~~~


**关键代码：通道上限向下对齐剪枝粒度**

来源：[scripts/prune_taylor_tiered_coco2017.py](../scripts/prune_taylor_tiered_coco2017.py)，cap_channels，第 84–85 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/prune_taylor_tiered_coco2017.py:84–85
# 对应：cap_channels
# 文件 SHA256：8840a14839da2a0a65eabe1daf6f42926240e1d3d35bbc3e500ffbe5aefef034
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

def cap_channels(width: int, fraction: float, step: int) -> int:
    return int(math.floor(width * fraction / step) * step)


**关键代码：在当前模型上重新计算Taylor分数**

来源：[scripts/prune_taylor_tiered_coco2017.py](../scripts/prune_taylor_tiered_coco2017.py)，局部代码段，第 136–139 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/prune_taylor_tiered_coco2017.py:136–139
# 对应：局部代码段
# 文件 SHA256：8840a14839da2a0a65eabe1daf6f42926240e1d3d35bbc3e500ffbe5aefef034
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

score_source = run_dir / f"taylor_step_{step:02d}.pt"
save_model(model, score_source, base)
scores, score_summary = collect_taylor_scores(score_source, calibration_yaml, active, device, args)
write_json(run_dir / f"taylor_step_{step:02d}.json", score_summary)


**结果**：14个low、13个medium、8个high；接受24步。GMAC 10.7991→10.2251（−5.32%），参数−3.86%，raw val=0.4181。5%指计算量预算，不代表每层或总参数都减5%。

来源：[实验17报告](../reports/experiment17_tiered_taylor_coco2017.md)。

### 14. 实验18：降低学习率也未必恢复得更好

**原理**：固定5%结构，用AdamW、lr=1e-4、5轮、BN更新、无增强训练。把raw加入候选，防止训练“best”仍不如训练前。

~~~bash
# 来源：scripts/recover_tiered_coco2017.py -> parse_args
STAGE5=runs/prune/experiment17_tiered_taylor_coco2017/20260915_123525
"$PY" scripts/recover_tiered_coco2017.py --dataset-root "$DATA" --pruning-run "$STAGE5" --sensitivity-run "$SENS" --epochs 5 --batch 128 --device 0 --execute
~~~

**结果**：同一tune上raw=0.514366，best/last均0.489790，选回raw；选中完整val=0.418059。不能拿tune的0.514366减val的0.418059，说成一次训练掉了这些点。

来源：[实验18报告](../reports/experiment18_tiered_taylor_recovery.md)。


<a id="chapter-4"></a>

## 四、BN校准与嵌套剪枝阶梯（19A）

### 15. BN校准到底更新什么

**原理**：剪枝改变激活分布，原BN运行均值/方差可能不匹配。冻结所有参数，仅BN处于train状态，图像前向更新统计。

运行均值可理解为new=(1−momentum)×old+momentum×batch_mean。momentum是新batch统计的权重；0.005表示缓慢更新，不是优化器学习率。reset=False保留原有统计。

~~~bash
# 来源：scripts/recalibrate_bn_coco2017.py -> parse_args / main
"$PY" scripts/recalibrate_bn_coco2017.py --dataset-root "$DATA" --pruning-run "$STAGE5" --sensitivity-run "$SENS" --batch 128 --device 0 --extended --execute
~~~

--extended扫描多种图像数/momentum，不是只跑最终8192张方案。当前内部部分位置固定cuda:0，此处使用device=0。


**关键代码：冻结参数，仅打开BN统计更新**

来源：[scripts/recalibrate_bn_coco2017.py](../scripts/recalibrate_bn_coco2017.py)，局部代码段，第 47–64 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/recalibrate_bn_coco2017.py:47–64
# 对应：局部代码段
# 文件 SHA256：2f832d5aa317394c8c0a3500e4e9f1d21233b6de0bb36d7d4bbdd5f8564cd6a0
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

model.eval()
bn_count = 0
for parameter in model.parameters():
    parameter.requires_grad_(False)
for module in model.modules():
    if isinstance(module, nn.BatchNorm2d):
        bn_count += 1
        if reset:
            module.reset_running_stats()
        module.momentum = momentum
        module.train()
seen = 0
started = time.perf_counter()
with torch.inference_mode():
    for batch in loader:
        tensor = batch["img"].to(device, non_blocking=True).float() / 255.0
        model(tensor)
        seen += tensor.shape[0]


最终四节点统一采用8192张、momentum=0.005、不重置的BN版source。脚本检查仅BN buffers改变，保存重载不改变结构。


**BN收益表：只在同一tune集合内比较。**

| 节点 | raw tune | BN/source tune | BN提升（tune） | source val |
|---:|---:|---:|---:|---:|
| 5 | 0.514366 | 0.522828 | +0.008462 | 0.425604 |
| 10 | 0.431151 | 0.468309 | +0.037159 | 0.386686 |
| 15 | 0.195039 | 0.311249 | +0.116210 | 0.258892 |
| 20 | 0.053326 | 0.135618 | +0.082292 | 0.117799 |

数据来源：[最终核验报告](../reports/experiment20_COCO2017final_comparison.md) §5. BN收益与训练前基线；历史结果转录，本次未重新运行训练或评估。


node15的tune提升0.116210，不能写成完整val提升11.621 AP点：10/15/20%的raw没有完整val记录。BN适配统计分布，没有通过检测损失更新卷积权重。

### 16. node5 / node10 / node15 / node20：一条嵌套路径

**原理**：实验17的5% raw继续剪到10%、15%、20%。每步重新打分，已删通道不加回；每到节点保存副本做BN校准和评价。BN source不返回raw主链。

每个节点从官方模型独立重剪，或将恢复后的权重送回主链，都改变了本实验定义。

~~~bash
# 来源：scripts/prune_taylor_ladder_coco2017.py -> parse_args / main
"$PY" scripts/prune_taylor_ladder_coco2017.py --dataset-root "$DATA" --stage5-run "$STAGE5" --sensitivity-run "$SENS" --channel-step 8 --calibration-batch 32 --bn-batch 128 --val-batch 128 --device 0 --execute
~~~

内部固定后续目标10/15/20%，没有--target-reduction参数；实际运行目录名带experiment20。


**关键代码：各阶段的敏感度档位累计上限**

来源：[scripts/prune_taylor_ladder_coco2017.py](../scripts/prune_taylor_ladder_coco2017.py)，局部代码段，第 32–36 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/prune_taylor_ladder_coco2017.py:32–36
# 对应：局部代码段
# 文件 SHA256：20ce47e803e86c2e4882038e48cf5ada9627aa2d2ca02875180f13b5c3fa725e
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

CAPS = {
    0.10: {"low": 0.25, "medium": 0.125, "high": 0.0625},
    0.15: {"low": 0.375, "medium": 0.25, "high": 0.125},
    0.20: {"low": 0.50, "medium": 0.375, "high": 0.25},
}


**关键代码：保存raw，在独立加载的副本上校准并按tune选择source**

来源：[scripts/prune_taylor_ladder_coco2017.py](../scripts/prune_taylor_ladder_coco2017.py)，endpoint，第 68–85 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/prune_taylor_ladder_coco2017.py:68–85
# 对应：endpoint
# 文件 SHA256：20ce47e803e86c2e4882038e48cf5ada9627aa2d2ca02875180f13b5c3fa725e
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

def endpoint(model: nn.Module, template: Path, target: float, run_dir: Path,
             calibration8: Path, tune: Path, full: Path, args, eval_args) -> dict[str, Any]:
    label = f"stage_{int(target * 100):02d}"
    raw = run_dir / label / "raw.pt"
    raw.parent.mkdir(parents=True, exist_ok=True)
    save_model(model, raw, template)
    raw_tune = evaluate(raw, tune, run_dir / label / "validation_tune_raw", eval_args)
    bn = run_dir / label / "bn_m0005_8192.pt"
    bn_details = recalibrate(raw, calibration8, bn, 8192, args.bn_batch, args.workers,
                             torch.device("cuda:0"), args.seed, 0.005, False)
    bn_tune = evaluate(bn, tune, run_dir / label / "validation_tune_bn", eval_args)
    selected = bn if bn_tune["map50_95"] >= raw_tune["map50_95"] else raw
    selected_label = "bn_m0005_8192" if selected == bn else "raw"
    full_metrics = evaluate(selected, full, run_dir / label / "validation_full_selected", eval_args)
    return {"target": target, "raw": {"path": str(raw), "sha256": sha256(raw), "tune": raw_tune},
            "bn": {"path": str(bn), "sha256": sha256(bn), "tune": bn_tune, "details": bn_details},
            "selected": {"label": selected_label, "path": str(selected), "sha256": sha256(selected),
                         "tune": bn_tune if selected == bn else raw_tune, "full_val": full_metrics}}


**计划与实际的差别**：计划逐级开放层级；代码每档直接纳入所有未达上限候选。实际每到节点就做BN/评价，而非全部剪完后才做；raw隔离仍保持。不能把这些偏差改写成严格符合计划。

5%原有24步，后续三阶段共96个接受动作。来源：[最终核验报告 §2](../reports/experiment20_COCO2017final_comparison.md)。


<a id="chapter-5"></a>

## 五、普通恢复与YOLO教师蒸馏（19B/19C）

### 17. control：先看普通训练能恢复多少

**原理**：检测标签训练让剩余参数适应剪后结构。比较KD时固定source、数据、epoch、物理batch、优化器和增强。

12任务统一20轮、640输入、batch=nbs=128、AdamW、lr0=0.001、lrf=0.1、学习率warmup=1轮、weight_decay=0.0005、AMP、seed=42。mosaic=0.5，最后10轮关闭mosaic。与实验18低学习率/无增强配方不同。


**关键代码：control的标准恢复配方**

来源：[scripts/recover_ladder_control_coco2017.py](../scripts/recover_ladder_control_coco2017.py)，局部代码段，第 52–66 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/recover_ladder_control_coco2017.py:52–66
# 对应：局部代码段
# 文件 SHA256：2894fa56264293afed4c810e55893e53521078a74a9cb1082eba9541c88a6847
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

PRESET = {
    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.1,
    "warmup_epochs": 1.0,
    "warmup_bias_lr": 0.01,
    "weight_decay": 0.0005,
    "mosaic": 0.5,
    "fliplr": 0.5,
    "scale": 0.3,
    "translate": 0.1,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
}


**关键代码：直接交给训练器已剪模型，并检查结构未被重建**

来源：[scripts/recover_ladder_control_coco2017.py](../scripts/recover_ladder_control_coco2017.py)，局部代码段，第 224–235 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/recover_ladder_control_coco2017.py:224–235
# 对应：局部代码段
# 文件 SHA256：2894fa56264293afed4c810e55893e53521078a74a9cb1082eba9541c88a6847
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

trainer = DetectionTrainer(overrides=overrides)
# 直接放入已加载的 pruned model，保留结构化剪枝后的通道宽度。
trainer.model = source_model.model

def check_final(trainer_instance: DetectionTrainer) -> None:
    if model_shapes(trainer_instance.model) != expected_shapes:
        raise RuntimeError("Recovery changed the model architecture")

trainer.callbacks["on_train_end"].append(check_final)

torch.cuda.reset_peak_memory_stats(int(args.device))
trainer.train()


trainer.model = source_model.model 保留已剪宽度；仅按官方YAML重新建模可能恢复原结构。比较前后形状是确认“仍是同一个学生”的关键。

### 18. node15单任务：预检、control、YOLO11s KD

以下使用核验过的历史source。新生成source不一定有相同文件哈希；恢复脚本对各node硬编码期望SHA256，不能为了跑通随意关闭。新实验应建立自己的清单和谱系。

~~~bash
# 来源：scripts/run_experiment19b.sh:13–35及恢复脚本parse_args
SRC15=runs/prune/experiment20_taylor_ladder_coco2017/20260917_023832/stage_15/bn_m0005_8192.pt
RECOVERY="$SENS/recovery.yaml"

# 预检：检查文件、哈希、数据和模型信息
"$PY" scripts/recover_ladder_control_coco2017.py --node 15 --source "$SRC15" --data "$RECOVERY" --val-data configs/coco2017.yaml --dataset-root "$DATA"

# 普通恢复，正式20轮
"$PY" scripts/recover_ladder_control_coco2017.py --node 15 --source "$SRC15" --data "$RECOVERY" --val-data configs/coco2017.yaml --dataset-root "$DATA" --epochs 20 --batch 128 --nbs 128 --device 0 --execute

# 同一source + 官方YOLO11s教师
"$PY" scripts/recover_ladder_yolo_kd_coco2017.py --node 15 --source "$SRC15" --teacher weights/yolo11s.pt --teacher-name yolo11s --data "$RECOVERY" --val-data configs/coco2017.yaml --dataset-root "$DATA" --epochs 20 --batch 128 --nbs 128 --kd-weight 0.5 --kd-warmup 2 --device 0 --execute
~~~

PowerShell同类预检（本地需同一source、数据及可解析清单）：

~~~powershell
# 来源：scripts/recover_ladder_control_coco2017.py -> parse_args
$source15 = '.\runs\prune\experiment20_taylor_ladder_coco2017\20260917_023832\stage_15\bn_m0005_8192.pt'
$recoveryYaml = '.\runs\analysis\experiment16_taylor_sensitivity_coco2017\20260915_122417\recovery.yaml'
.\.venv\Scripts\python.exe scripts\recover_ladder_control_coco2017.py --node 15 --source $source15 --data $recoveryYaml --val-data configs/coco2017.yaml --dataset-root 'C:\Users\22565\datasets\coco'
~~~

服务器权重或/root/...清单未迁移到本地时，预检会失败；只改dataset-root不保证清单内容自动转换。

### 19. YOLO11s / YOLO11m特征教师：四步对齐

**原理**：同一图像送学生和冻结教师；读Detect输入来源并hook提取P3/P4/P5；1×1卷积将学生通道映射到教师通道，空间插值，再计算余弦距离。

P3/P4/P5对应stride 8/16/32，640输入时通常为80×80、40×40、20×20。投影学习“怎样比较特征”，不永久扩大部署学生。

每尺度损失为通道归一化后的1−cosine similarity，在batch/空间平均；层权重0.25/0.50/0.25。教师eval且无梯度前向；学生与投影层接受梯度。


**关键代码：动态读Detect输入，不猜网络层号**

来源：[scripts/recover_ladder_yolo_kd_coco2017.py](../scripts/recover_ladder_yolo_kd_coco2017.py)，_find_pyramid_layers，第 304–309 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/recover_ladder_yolo_kd_coco2017.py:304–309
# 对应：_find_pyramid_layers
# 文件 SHA256：64a23544f9cb8a8bf1ebe78cedc3419a866048392c97045f1c30b6bb64521804
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

def _find_pyramid_layers(model: nn.Module) -> list[nn.Module]:
    detect = model.model[-1]
    sources = getattr(detect, "f", None)
    if not isinstance(sources, (list, tuple)) or len(sources) != 3:
        raise RuntimeError("Could not identify Detect's P3/P4/P5 input layers")
    return [model.model[index] for index in sources]


**关键代码：冻结教师、探测通道、投影和余弦损失**

来源：[scripts/recover_ladder_yolo_kd_coco2017.py](../scripts/recover_ladder_yolo_kd_coco2017.py)，YoloFeatureDistiller，第 217–267 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/recover_ladder_yolo_kd_coco2017.py:217–267
# 对应：YoloFeatureDistiller
# 文件 SHA256：64a23544f9cb8a8bf1ebe78cedc3419a866048392c97045f1c30b6bb64521804
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

class YoloFeatureDistiller(nn.Module):
    """冻结 YOLO 教师 + 可训练 P3/P4/P5 1×1 投影头。"""

    def __init__(self, teacher: nn.Module, student_channels: list[int], device: torch.device, weight: float):
        super().__init__()
        self.base_weight = weight
        self.current_weight = 0.0
        self.level_weights = (0.25, 0.50, 0.25)
        self.teacher = teacher.eval()
        for parameter in self.teacher.parameters():
            parameter.requires_grad_(False)

        detect = self.teacher.model[-1]
        sources = getattr(detect, "f", None)
        if not isinstance(sources, (list, tuple)) or len(sources) != 3:
            raise RuntimeError("Teacher Detect does not expose P3/P4/P5 sources")
        self.teacher_layers = [self.teacher.model[index] for index in sources]
        self.teacher_features: dict[str, torch.Tensor] = {}
        for level, layer in zip(("p3", "p4", "p5"), self.teacher_layers):
            layer.register_forward_hook(_CaptureFeature(self.teacher_features, level))

        teacher_channels: list[int] = []
        with torch.no_grad():
            self.teacher(torch.zeros(1, 3, 640, 640, device=device))
            for level in ("p3", "p4", "p5"):
                teacher_channels.append(self.teacher_features[level].shape[1])
        self.teacher_features.clear()

        self.projectors = nn.ModuleList(
            nn.Conv2d(sc, tc, kernel_size=1, bias=False)
            for sc, tc in zip(student_channels, teacher_channels)
        ).to(device)
        for projector in self.projectors:
            nn.init.kaiming_normal_(projector.weight, mode="fan_out", nonlinearity="linear")

    @torch.no_grad()
    def teacher_pyramid(self, images: torch.Tensor) -> list[torch.Tensor]:
        self.teacher_features.clear()
        self.teacher(images)
        return [self.teacher_features[level] for level in ("p3", "p4", "p5")]

    def loss(self, student_features: list[torch.Tensor], images: torch.Tensor) -> tuple[torch.Tensor, list[torch.Tensor]]:
        teacher_features = self.teacher_pyramid(images)
        level_losses = []
        for student, teacher, projector in zip(student_features, teacher_features, self.projectors):
            teacher = F.interpolate(teacher, size=student.shape[-2:], mode="bilinear", align_corners=False)
            projected = F.normalize(projector(student), dim=1)
            teacher = F.normalize(teacher, dim=1)
            level_losses.append(1.0 - (projected * teacher).sum(dim=1).mean())
        total = sum(weight * loss for weight, loss in zip(self.level_weights, level_losses))
        return total, level_losses


### 20. 总损失、batch缩放与两轮升权

概念上L_total=L_det+λ(t)×L_KD。L_det包含box/cls/dfl。项目为了适配当前Ultralytics版本，将KD项按batch大小缩放后拼入损失向量。这是本项目版本的实现，不应直接照搬到其他版本。

kd-weight=0.5是最终权重；kd-warmup=2时按(epoch+1)/2升权，第一轮0.25，第二轮起0.5。学习率warmup_epochs=1是另一件事。


**关键代码：KD项加入检测损失向量**

来源：[scripts/recover_ladder_yolo_kd_coco2017.py](../scripts/recover_ladder_yolo_kd_coco2017.py)，YoloKDStudentModel，第 270–287 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/recover_ladder_yolo_kd_coco2017.py:270–287
# 对应：YoloKDStudentModel
# 文件 SHA256：64a23544f9cb8a8bf1ebe78cedc3419a866048392c97045f1c30b6bb64521804
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

class YoloKDStudentModel(DetectionModel):
    """DetectionModel 附加 P3/P4/P5 YOLO 教师特征损失。"""

    def loss(self, batch: dict, preds=None):
        regular_loss, loss_items = super().loss(batch, preds)
        if not self.training:
            return regular_loss, loss_items
        features = self._kd_student_features
        if any(level not in features for level in ("p3", "p4", "p5")):
            raise RuntimeError("P3/P4/P5 hooks did not capture all student features")
        kd_loss, level_losses = self._kd_distiller.loss(
            [features["p3"], features["p4"], features["p5"]], batch["img"]
        )
        loss_items["kd_loss"] = kd_loss.detach()
        for level, level_loss in zip(("p3", "p4", "p5"), level_losses):
            loss_items[f"kd_{level}"] = level_loss.detach()
        scaled = kd_loss * self._kd_distiller.current_weight * batch["img"].shape[0]
        return torch.cat((regular_loss, scaled.reshape(1))), loss_items


**关键代码：训练前初始化及每轮设置KD权重**

来源：[scripts/recover_ladder_yolo_kd_coco2017.py](../scripts/recover_ladder_yolo_kd_coco2017.py)，局部代码段，第 412–418 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/recover_ladder_yolo_kd_coco2017.py:412–418
# 对应：局部代码段
# 文件 SHA256：64a23544f9cb8a8bf1ebe78cedc3419a866048392c97045f1c30b6bb64521804
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

def prepare(trainer_instance: DetectionTrainer) -> None:
    # on_pretrain_routine_end 阶段 trainer.epoch 尚未赋值，不能读 epoch。
    trainer_instance.model._kd_distiller.current_weight = args.kd_weight / max(1, args.kd_warmup)

def set_weight(trainer_instance: DetectionTrainer) -> None:
    fraction = min(1.0, (trainer_instance.epoch + 1) / max(1, args.kd_warmup))
    trainer_instance.model._kd_distiller.current_weight = args.kd_weight * fraction


**关键代码：切换训练模式、构建优化器时重新冻结教师**

来源：[scripts/recover_ladder_yolo_kd_coco2017.py](../scripts/recover_ladder_yolo_kd_coco2017.py)，KDTrainer，第 362–378 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/recover_ladder_yolo_kd_coco2017.py:362–378
# 对应：KDTrainer
# 文件 SHA256：64a23544f9cb8a8bf1ebe78cedc3419a866048392c97045f1c30b6bb64521804
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

class KDTrainer(DetectionTrainer):
    """标准训练 + 每次 build_optimizer / 进入 train 时重新冻结教师。"""

    def _freeze_teacher(self) -> None:
        distiller = getattr(self.model, "_kd_distiller", None)
        if distiller is not None:
            distiller.teacher.eval()
            for parameter in distiller.teacher.parameters():
                parameter.requires_grad_(False)

    def build_optimizer(self, *args, **kwargs):
        self._freeze_teacher()
        return super().build_optimizer(*args, **kwargs)

    def _model_train(self):
        super()._model_train()
        self._freeze_teacher()


初始化freeze一次不够：训练器可能重切train状态或开启梯度，所以再次保证教师固定。学生BN按正式恢复配方更新，不能直接照搬COCO128笔记的冻结学生BN。

**YOLO11m教师命令**：

~~~bash
# 来源：scripts/run_experiment19c.sh:37–44
"$PY" scripts/recover_ladder_yolo_kd_coco2017.py --node 15 --source "$SRC15" --teacher yolo11m.pt --teacher-name yolo11m --data "$RECOVERY" --dataset-root "$DATA" --epochs 20 --batch 128 --nbs 128 --device 0 --execute
~~~

历史YOLO11m权重在仓库根目录，按teacher-name核对哈希。任务输出目录仍叫experiment19b，要按teacher配置识别19C任务。


<a id="chapter-6"></a>

## 六、DINOv2跨架构教师与部署权重

### 21. DINOv2为什么需要额外对齐

**原理**：ViT-S/14是视觉Transformer，YOLO是检测网络，这是跨架构视觉蒸馏。DINOv2没有与YOLO同构的COCO检测输出，项目比较中间视觉特征。

| DINO侧 | 学生侧 | 对齐操作 |
|---|---|---|
| 第4/8/12个Block（索引3/7/11） | P3/P4/P5 | 一次教师前向取三层 |
| 384通道 | 剪后实际通道数 | 1×1投影到384 |
| 644/14=46，46×46网格 | 通常80/40/20网格 | 教师特征插值到学生大小 |
| DINO归一化 | YOLO输入0–1 | 教师支路额外减mean、除std |

640不能被14整除，教师支路插值到644；学生仍640。层权重和总KD权重与YOLO教师方案一致。

~~~bash
# 来源：scripts/run_experiment19c.sh:47–54
"$PY" scripts/recover_ladder_dinov2_kd_coco2017.py --node 15 --source "$SRC15" --data "$RECOVERY" --dataset-root "$DATA" --epochs 20 --batch 128 --nbs 128 --kd-weight 0.5 --kd-warmup 2 --dino-size 644 --device 0 --execute
~~~

教师由torch.hub加载，缓存缺失时涉及下载；source仍需匹配node哈希。


**关键代码：DINO输入归一化、多层特征提取与对齐**

来源：[scripts/recover_ladder_dinov2_kd_coco2017.py](../scripts/recover_ladder_dinov2_kd_coco2017.py)，MultiScaleDINOFeatureDistiller，第 218–260 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/recover_ladder_dinov2_kd_coco2017.py:218–260
# 对应：MultiScaleDINOFeatureDistiller
# 文件 SHA256：9f4501752613a8380fbc948901dace0d8f2956881a061f5da22be553a94367ed
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

class MultiScaleDINOFeatureDistiller(nn.Module):
    """一次前向取 DINOv2 块 4/8/12，对齐 YOLO P3/P4/P5。"""

    def __init__(self, student_channels: list[int], device: torch.device, dino_size: int, weight: float):
        super().__init__()
        if dino_size % 14:
            raise ValueError("--dino-size must be divisible by DINOv2 patch size 14")
        if len(student_channels) != 3:
            raise ValueError("P3/P4/P5 distillation requires exactly three student features")
        self.dino_size = dino_size
        self.base_weight = weight
        self.current_weight = 0.0
        self.block_indices = (3, 7, 11)  # zero-based：transformer blocks 4/8/12
        self.level_weights = (0.25, 0.50, 0.25)
        self.teacher = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14").to(device).eval()
        for parameter in self.teacher.parameters():
            parameter.requires_grad_(False)
        self.projectors = nn.ModuleList(
            nn.Conv2d(channels, 384, kernel_size=1, bias=False) for channels in student_channels
        ).to(device)
        for projector in self.projectors:
            nn.init.kaiming_normal_(projector.weight, mode="fan_out", nonlinearity="linear")
        self.register_buffer("mean", torch.tensor(DINO_MEAN, device=device).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("std", torch.tensor(DINO_STD, device=device).view(1, 3, 1, 1), persistent=False)

    @torch.no_grad()
    def teacher_features(self, images: torch.Tensor) -> tuple[torch.Tensor, ...]:
        images = F.interpolate(images, size=(self.dino_size, self.dino_size), mode="bilinear", align_corners=False)
        normalized = (images - self.mean) / self.std
        return self.teacher.get_intermediate_layers(
            normalized, n=self.block_indices, reshape=True, norm=True
        )

    def loss(self, student_features: list[torch.Tensor], images: torch.Tensor) -> tuple[torch.Tensor, list[torch.Tensor]]:
        teacher_features = self.teacher_features(images)
        level_losses = []
        for student, teacher, projector in zip(student_features, teacher_features, self.projectors):
            teacher = F.interpolate(teacher, size=student.shape[-2:], mode="bilinear", align_corners=False)
            projected = F.normalize(projector(student), dim=1)
            teacher = F.normalize(teacher, dim=1)
            level_losses.append(1.0 - (projected * teacher).sum(dim=1).mean())
        total = sum(level_weight * loss for level_weight, loss in zip(self.level_weights, level_losses))
        return total, level_losses


### 22. 训练权重为什么要清理

训练wrapper含学生、教师、projector、hook和训练状态。直接计数或部署会混入训练期模块，还可能依赖自定义类而加载失败。

清理：复制模型→删除捕获hook和字典→删除蒸馏模块→恢复DetectionModel→保存clean权重。只加载可信项目检查点；weights_only=False的pickle加载会恢复Python对象。


**关键代码：移除教师与投影，恢复普通YOLO学生**

来源：[scripts/recover_ladder_yolo_kd_coco2017.py](../scripts/recover_ladder_yolo_kd_coco2017.py)，detach_for_inference，第 340–351 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/recover_ladder_yolo_kd_coco2017.py:340–351
# 对应：detach_for_inference
# 文件 SHA256：64a23544f9cb8a8bf1ebe78cedc3419a866048392c97045f1c30b6bb64521804
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

def detach_for_inference(model: nn.Module) -> nn.Module:
    """去除训练期教师/投影，恢复普通 YOLO 学生。"""
    clean = copy.deepcopy(model).float().eval()
    for module in clean.modules():
        for hook_id, hook in list(module._forward_hooks.items()):
            if isinstance(hook, _CaptureFeature):
                del module._forward_hooks[hook_id]
    clean.__dict__.pop("_kd_student_features", None)
    if "_kd_distiller" in clean._modules:
        del clean._modules["_kd_distiller"]
    clean.__class__ = DetectionModel
    return clean


DINO对应实现为recover_ladder_dinov2_kd_coco2017.py的detach_for_inference()。最终核验确认选中学生与source形状、参数量及双口径GMAC一致，无教师/投影残留。


<a id="chapter-7"></a>

## 七、结果怎样读：选择、增益归因与速度

### 23. 为什么将source加入候选

训练器best只代表训练轨迹内的最佳候选，不自动与训练前比较。重新在同一tune测source/best/last，选最高者，再测完整val。


**关键代码：按tune选择，再测选中权重的val2017**

来源：[scripts/recover_ladder_control_coco2017.py](../scripts/recover_ladder_control_coco2017.py)，局部代码段，第 344–354 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/recover_ladder_control_coco2017.py:344–354
# 对应：局部代码段
# 文件 SHA256：2894fa56264293afed4c810e55893e53521078a74a9cb1082eba9541c88a6847
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

candidates = [
    {"checkpoint": "source", "tune": source_tune, "weights": source},
    {"checkpoint": "best", "tune": best_tune, "weights": best},
    {"checkpoint": "last", "tune": last_tune, "weights": last},
]
winner = max(candidates, key=lambda row: row["tune"]["map50_95"])

# 4) 选定 checkpoint 在 val2017 上最终评价
val_final = evaluate(
    winner["weights"], Path(val_data_info["runtime"]), run_dir / "validation" / "val2017_selected", args
)


### 24. 正式12任务：训练完成不等于训练后权重胜出

| 节点 | 方案 | 训练后最高tune | 选中 | 选中tune | 选中val | val相对source | val相对control |
|---:|---|---:|---|---:|---:|---:|---:|
| 5 | control | 0.454161 | source | 0.522828 | 0.425604 | +0.000000 | +0.000000 |
| 5 | YOLO11s KD | 0.453540 | source | 0.522828 | 0.425604 | +0.000000 | +0.000000 |
| 10 | control | 0.449928 | source | 0.468309 | 0.386686 | +0.000000 | +0.000000 |
| 10 | YOLO11s KD | 0.456338 | source | 0.468309 | 0.386686 | +0.000000 | +0.000000 |
| 15 | control | 0.448279 | best | 0.448279 | 0.423467 | +0.164575 | +0.000000 |
| 15 | YOLO11s KD | 0.451473 | best | 0.451473 | 0.427374 | +0.168482 | +0.003907 |
| 15 | YOLO11m KD | 0.452120 | best | 0.452120 | 0.424587 | +0.165695 | +0.001120 |
| 15 | DINOv2 KD | 0.445610 | best | 0.445610 | 0.423965 | +0.165073 | +0.000498 |
| 20 | control | 0.449289 | best | 0.449289 | 0.421530 | +0.303731 | +0.000000 |
| 20 | YOLO11s KD | 0.443622 | best | 0.443622 | 0.422572 | +0.304773 | +0.001042 |
| 20 | YOLO11m KD | 0.448004 | best | 0.448004 | 0.421673 | +0.303874 | +0.000143 |
| 20 | DINOv2 KD | 0.446232 | best | 0.446232 | 0.421245 | +0.303447 | -0.000284 |

数据来源：[最终核验报告](../reports/experiment20_COCO2017final_comparison.md) §6. 12个正式任务完整结果；历史结果转录，本次未重新运行训练或评估。


5%/10%各方案都完成20轮，但tune不如source而选回source。相同val不是“两种训练完全一样”，而是最终指向同一文件。未选best/last没有完整val，不能编造。

### 25. 恢复收益与蒸馏收益分开算

| 节点 | source val | control val | YOLO11s KD val | 普通恢复（AP点） | KD额外（AP点） |
|---|---:|---:|---:|---:|---:|
| node15 | 0.258892 | 0.423467 | 0.427374 | +16.4575 | +0.3907 |
| node20 | 0.117799 | 0.421530 | 0.422572 | +30.3731 | +0.1042 |

来源：最终报告§6，均为同一val内差值。不能把node20从约0.12到0.42的全部变化归于蒸馏。

本次YOLO11s KD的val最高，不能据此证明它普遍优于其他教师。优化难度、特征不匹配等只是解释假设，没有消融/重复验证。DINO在node15相对control极小正差、node20略负，也不应概括为“完全无效”。


### 26. 每种计数工具使用自己的基线

| 模型 | 参数量 | TP GMAC | TP降幅 | THOP GMAC | THOP降幅 |
|---|---:|---:|---:|---:|---:|
| 官方YOLO11s | 9,458,752 | 10.7990592 | 0.00% | 10.8567296 | 0.00% |
| node5 | 9,093,536 | 10.2250816 | 5.32% | 10.2810624 | 5.30% |
| node10 | 8,549,624 | 9.7159040 | 10.03% | 9.7706880 | 10.00% |
| node15 | 8,063,896 | 9.1649568 | 15.13% | 9.2184224 | 15.09% |
| node20 | 7,474,088 | 8.6322592 | 20.06% | 8.6844000 | 20.01% |

数据来源：[最终核验报告](../reports/experiment20_COCO2017final_comparison.md) §4. 统一计算量口径；历史结果转录，本次未重新运行训练或评估。


node15参数减少14.75%、GMAC减少15.13%；node20分别20.98%和20.06%。旧报告曾混算TP官方与THOP学生，最终报告已纠正。以下stats函数是TP口径，其他model_stats等入口可能用THOP，应读具体调用。


**关键代码：Torch-Pruning计数并转换为GMAC**

来源：[scripts/prune_greedy_coco2017.py](../scripts/prune_greedy_coco2017.py)，stats，第 92–95 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：scripts/prune_greedy_coco2017.py:92–95
# 对应：stats
# 文件 SHA256：f271b956d3932569700e86e8766052583fa39f62fda5dada655c49740789ab09
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

def stats(model, example):
    # TP's operation counter changes training flags on its copy, not the source.
    macs, _ = tp.utils.count_ops_and_params(copy.deepcopy(model).eval(), example)
    return sum(p.numel() for p in model.parameters()), float(macs) / 1e9


### 27. Pareto选型与验证集边界

在val越高越好、GMAC越低越好的坐标上，node15优于本次node5/node10；node20再省计算但精度略低。

节点与教师最终排序看过val，val参与了事后选型，不能再称完全独立于选型的最终测试。tune排序确实不同：node15的YOLO11m为0.452120，高于YOLO11s的0.451473；node20 control为0.449289，高于YOLO11s的0.443622。

### 28. GMAC降了，为什么没有明显加速

最终复测：A800、batch=1、640×640、默认FP32、eval/no_grad、全零输入、30次预热、200次前向，每次前后CUDA同步。没有显式融合、autocast、编译或TensorRT，不含读图、传输、预处理和NMS。


**同一协议下的短前向延迟。**

| 模型 | 历史mean ms（旧报告） | 历史median ms | 本次mean ms | 本次median ms | 本次P90 ms |
|---|---:|---:|---:|---:|---:|
| yolo11s_official | 7.57 | 7.48 | 7.86 | 7.80 | 7.90 |
| node15_15pct | 7.50 | 7.48 | 7.76 | 7.73 | 7.78 |
| node20_20pct | 7.54 | 7.49 | 7.99 | 7.92 | 8.10 |

数据来源：[最终核验报告](../reports/experiment20_COCO2017final_comparison.md) §8. 延迟：历史值和本次复测；历史结果转录，本次未重新运行训练或评估。


**关键代码：原延迟脚本：预热、同步与计时**

来源：[reports/experiment19_audit/20260918_101935/server/exp19c_latency.py](../reports/experiment19_audit/20260918_101935/server/exp19c_latency.py)，bench，第 4–31 行。以下逐字摘录（仅统一缩进），依赖原脚本的导入和上下文，用于阅读，不是独立运行单元。


In [ ]:
# 来源：reports/experiment19_audit/20260918_101935/server/exp19c_latency.py:4–31
# 对应：bench
# 文件 SHA256：443589f7e869c6f4b60e5ce3e0f4ab4b340a997d28a5d46a663ea738eb04eb0d
# 阅读摘录：依赖原脚本上下文，不应直接 Run All。

def bench(weights, name, iters=200):
    model = YOLO(weights)
    m = model.model.eval().cuda()
    try:
        params = sum(p.numel() for p in m.parameters())
    except Exception:
        params = -1
    x = torch.zeros(1, 3, 640, 640, device="cuda")
    with torch.no_grad():
        for _ in range(30):
            m(x)
    torch.cuda.synchronize()
    times = []
    with torch.no_grad():
        for _ in range(iters):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            m(x)
            torch.cuda.synchronize()
            t1 = time.perf_counter()
            times.append((t1 - t0) * 1000.0)
    times.sort()
    mean = sum(times) / len(times)
    median = times[len(times)//2]
    p90 = times[int(len(times)*0.9)]
    print(f"{name}: params={params} mean={mean:.2f}ms median={median:.2f}ms p90={p90:.2f}ms min={times[0]:.2f}ms max={times[-1]:.2f}ms", flush=True)
    del model, m, x
    torch.cuda.empty_cache()


CUDA异步执行，计时区间需同步等GPU完成。200次前向用于统计同次测试分布，不是200次独立训练实验。

node15均值7.76ms与官方7.86ms接近，node20为7.99ms。本次未证实明显加速，不能认定瓶颈一定是访存或kernel启动，也不能保证换设备就更快。

~~~bash
# 来源：审计附件中的原延迟脚本
# 内部使用历史服务器绝对权重路径，需文件仍在原位置
"$PY" reports/experiment19_audit/20260918_101935/server/exp19c_latency.py
~~~

### 29. 训练成本也要看

node15 control约1.445h；YOLO11s KD约2.077h/30.19GiB峰值allocated，YOLO11m约2.705h/33.12GiB，DINOv2约3.117h/35.60GiB。教师与对齐操作增加训练成本，清理后部署不保留教师开销。

时间来自results.csv累计time，含训练循环验证，不是纯GPU核时间；并行任务不能相加当墙钟时间。control峰值显存缺失，不能推算。完整表见最终报告§9。


<a id="chapter-8"></a>

## 八、终端复现、文件结构与排错

### 30. 参数怎样改

| 想研究的变化 | 入口/参数 | 对照中需要固定 |
|---|---|---|
| 独立Taylor预算 | prune_taylor_coco2017.py / --target-reduction | 父模型、数据、评分设置 |
| 每步剪多少 | --channel-step | 宽度约束和目标GMAC |
| 分档上限 | tiered的--low-cap/--medium-cap；ladder的CAPS | 敏感度排序、同源raw链 |
| 恢复多久 | --epochs | source、数据、优化器、增强 |
| KD强度 | --kd-weight、--kd-warmup | source、教师、恢复配方 |
| 学习率/增强 | 各恢复脚本PRESET | 同时检查control和KD脚本 |
| DINO尺寸 | --dino-size | 能被14整除，记录计算成本 |

nbs不等于物理batch，本次二者都128。调小batch适应显存属于新配方，仅改nbs不能默认所有细节等价。以上是入口说明，不是已经完成的新消融。

### 31. 调度与进度检查

历史run_experiment19b.sh假设node5 control已单独启动，只安排剩余7任务，不是从零跑齐8任务。19C的两个节点在GPU0/1各跑两种教师。调度器写死服务器路径，最后的ALL DONE不代替逐任务核验。

~~~bash
# 来源：scripts/run_experiment19c.sh:3、57–73
# 需GPU0/1、source、教师、数据齐全；此命令启动四个正式任务
bash scripts/run_experiment19c.sh 15 20

# 整理新增的只读检查
nvidia-smi
tail -n 30 exp19c_scheduler.log
find runs/recovery -name results.csv -print
RUN=runs/recovery/experiment19b_coco2017/node15_yolo11s_kd/20260917_101816
tail -n 5 "$RUN/training/results.csv"
cat "$RUN/comparison.csv"
~~~

~~~powershell
# 本地只读检查；路径来源为历史运行目录
Get-ChildItem -LiteralPath '.\runs\recovery' -Recurse -Filter results.csv
$run = '.\runs\recovery\experiment19b_coco2017\node15_yolo11s_kd\20260917_101816'
Get-Content -LiteralPath "$run\training\results.csv" -Tail 5
Import-Csv -LiteralPath "$run\comparison.csv"
~~~

### 32. 文件里分别看什么

~~~text
runs/.../<时间戳>/
├── run_info.json          状态、source哈希、配置、结果引用
├── run.log                异常栈和过程日志
├── candidates.csv         剪枝候选与拒绝原因（剪枝任务）
├── steps.json             已接受动作（部分剪枝入口）
├── comparison.csv         source/best/last与选中结果（恢复任务）
├── recovery_runtime.yaml  recovery训练、tune选择
├── val_runtime.yaml       完整val2017
└── training/
    ├── args.yaml          实际生效的训练参数
    ├── results.csv        每轮指标和累计时间
    └── weights/           best/last及KD的best_clean/last_clean
~~~

不同入口略有差异。入选文件路径/哈希见[checkpoint_manifest.csv](../reports/experiment19_audit/20260918_101935/checkpoint_manifest.csv)。记录SHA256、数据清单、实际args和结构签名，比只记best.pt可靠。

### 33. 常见问题与教训

| 现象 | 先检查 | 解释 |
|---|---|---|
| source哈希不匹配 | node、路径、manifest | 避免混入raw或其他剪枝路径 |
| 找不到数据 | YAML与txt清单的绝对路径 | 服务器清单不能只复制YAML迁移 |
| 通道宽度恢复了 | trainer.model、前后形状 | 可能重建了原结构 |
| 教师有梯度 | requires_grad、eval、优化器构建 | 检查重新freeze |
| hook缺特征 | Detect.f、特征键 | 按实际结构定位 |
| pickle找不到类 | wrapper类注册 | 部署优先用clean学生 |
| 训练后报告KeyError | results.csv、comparison.csv、args.yaml | 4个control曾补报告，不等于没训练 |
| best/source相同成绩 | selected与文件哈希 | 5%/10%最终回到source |
| 调度结束但缺结果 | 各任务状态、退出码、训练行数 | 不能只看调度日志结尾 |

本次整理现象和现有实现，没有修改训练脚本或重跑历史任务。

<a id="chapter-9"></a>

## 九、快速问答（自己复述原理）

**node15是不是第15层？** 它是累计GMAC减少约15%的阶段性模型；层位置用模块名和Detect.f定位。

**Taylor反向传播为什么不算训练？** 只计算梯度用于排序，没有optimizer.step，还检查权重和BN buffers没变。

**层敏感度与Taylor分数有何区别？** 敏感度通过试删后tune的mAP变化评价层；Taylor用检测损失梯度近似通道代价。前者用于层级约束，后者用于动作排序。

**BN校准为什么不用标签？** 前向激活即可更新统计，不需检测loss；Taylor评分需要检测标签。

**教师不更新，学生怎么学习？** 教师提供固定目标，特征差异的梯度经过投影层回到学生。

**为什么需要control？** source→KD包含普通训练恢复和KD额外作用，control帮助分开它们。

**为什么best不一定是最终模型？** 训练最佳可能不如source，最终候选还包含source与last。

**为什么大教师没有更好？** 本次val观察如此；特征不匹配等只是可能原因，缺重复/消融证明。

**省20%计算是否就快20%？** 不对应，实际A800 batch=1没明显加速，部署应按目标设备计时。

<a id="chapter-10"></a>

## 十、结论与下一步学习

### 34. 有证据支持的阶段结论

官方YOLO11s上的嵌套Taylor路径得到约5/10/15/20%计算量削减节点。BN改善统计适配；深剪枝经20轮恢复到约0.42 mAP，普通训练贡献主要恢复。YOLO11s KD在本次val额外增加约0.39/0.10 AP点。

node15+YOLO11s KD是本次val最高压缩候选0.427374，node20是较低GMAC候选0.422572；相对历史官方约0.4633仍低约3.59/4.07 AP点。没有无损压缩，未证实A800单图明显加速、教师普适最优或稳定显著KD收益。

结果是seed=42单次观察，无重复/置信区间；最终报告核对文件、结构、计数并短测延迟，但没重算COCO精度。计划有执行偏差；旧阶段报告写“待续”不代表现在没完成。

### 35. 建议学习顺序与研究方向（尚非结果）

1. 第一遍看主线图、术语和结果解释，口头讲清每次换方法的原因。
2. 第二遍跟collect_taylor_scores→prune_action→search/endpoint，画出raw与BN副本关系。
3. 第三遍跟train_control→YoloFeatureDistiller→loss→detach_for_inference，解释梯度路径与对照。
4. 第四遍手算node15计算量降幅、普通恢复增益、KD额外增益，检查数据集/工具口径。
5. 进一步研究先冻结选型规则，设计多种子与独立评价；部署固定设备、batch、精度和后处理再测。每次改变少量因素，保留新产物来源。

<a id="chapter-11"></a>

## 十一、报告与代码索引

| 阶段 | 报告 | 实现 |
|---|---|---|
| 07 | [官方基线](../reports/experiment07_yolo11s_coco2017_baseline.md) | [baseline](../scripts/run_yolo11s_coco2017_baseline.py) |
| 08 | 当前缺完成报告，按脚本及本地配置学习 | [train](../scripts/train_yolo11s_coco2017.py) |
| 09 | [敏感度](../reports/experiment09_yolo11s_coco2017_sensitivity.md) | [masking](../scripts/prune_sensitivity.py) |
| 10 | [贪心负结果](../reports/experiment10_coco2017_greedy.md) | [greedy](../scripts/prune_greedy_coco2017.py) |
| 14/15 | [谱系与诊断](../reports/experiment14_coco2017_recovery_plan.md) | [diagnose](../scripts/diagnose_coco2017_recovery.py)、[Taylor](../scripts/prune_taylor_coco2017.py) |
| 16 | [真实敏感度](../reports/experiment16_taylor_sensitivity_coco2017.md) | [sensitivity](../scripts/measure_taylor_sensitivity_coco2017.py) |
| 17 | [分档剪枝](../reports/experiment17_tiered_taylor_coco2017.md) | [tiered](../scripts/prune_taylor_tiered_coco2017.py) |
| 18 | [保守恢复](../reports/experiment18_tiered_taylor_recovery.md) | [recover](../scripts/recover_tiered_coco2017.py) |
| 19A | [计划](../reports/experiment19_pruning_ladder_distillation_plan.md)、[阶段记录](../reports/experiment19_pruning_ladder_distillation.md) | [BN](../scripts/recalibrate_bn_coco2017.py)、[ladder](../scripts/prune_taylor_ladder_coco2017.py) |
| 19B/19C | [最终核验](../reports/experiment20_COCO2017final_comparison.md) | [control](../scripts/recover_ladder_control_coco2017.py)、[YOLO KD](../scripts/recover_ladder_yolo_kd_coco2017.py)、[DINO KD](../scripts/recover_ladder_dinov2_kd_coco2017.py) |
| 审计 | [完整CSV](../reports/experiment19_audit/20260918_101935/comparison_full.csv)、[audit.json](../reports/experiment19_audit/20260918_101935/audit.json) | [延迟脚本](../reports/experiment19_audit/20260918_101935/server/exp19c_latency.py) |

行号对应本次读取的本地文件，每个代码单元还记录源文件SHA256。脚本改动后以函数名和哈希核对，避免将旧行号当成新版实现。
